# 🧠🔎 Agentic RAG: Retrieval-Augmented Generation with LangChain & Azure OpenAI 📚🤖

Welcome! In this notebook, you'll implement a retrieval agent—a system where an LLM can intelligently decide whether to fetch context from a vectorstore or respond directly to the user. Using LangChain and Azure OpenAI, you'll:

- 📄 Fetch and preprocess documents for retrieval
- 🗂️ Index documents for semantic search and create a retriever tool
- 🕹️ Build an agentic RAG workflow that decides when to use the retriever tool

This hands-on tutorial demonstrates how retrieval agents combine reasoning and context fetching to deliver accurate, context-aware answers. By the end, you'll understand how to orchestrate document ingestion, semantic search, and agentic decision-making for robust question answering

In [ ]:
%pip install  --quiet --upgrade pip
%pip install  --quiet yfinance 
%pip install  --quiet pandas
%pip install  --quiet langchain-community
%pip install  --quiet openai
%pip install  --quiet matplotlib
%pip install  --quiet langchain-openai
%pip install  --quiet langgraph
%pip install  --quiet tabulate

# Import Packages

In [ ]:
import sys
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools.retriever import create_retriever_tool
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langgraph.graph import MessagesState

### Azure OpenAI Chat Model Integration

This example demonstrates how to use the `AzureChatOpenAI` class from the `langchain_openai` package to interact with Azure OpenAI's GPT models.

#### Key Steps:
1. **Setup**: Import necessary libraries and retrieve credentials for the Azure OpenAI API.
2. **Credentials**: Ensure the Azure API key and endpoint are correctly set.
3. **Error Handling**: Check if the API key and endpoint are missing, and raise errors if needed.
4. **Chat Model**: Initialize the `AzureChatOpenAI` model by providing required credentials like API key, endpoint, model type, and API version.

#### Notes:
- Ensure that Azure OpenAI is set up and environment variables for the API key and endpoint are configured.


In [ ]:
import init_creds as creds

sys.path.insert(1, '../../')

# # Configure Azure OpenAI And OpenAI Embeddings
# AZURE_OPENAI_KEY = os.getenv('OPENAI_API_KEY')
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
# AZURE_OPENAI_API_VERSION = os.getenv('OPENAI_API_VERSION')


AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=AZURE_OPENAI_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION
    )

# Preprocess documents

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# Define the URLs to load
urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

# Load the documents
docs = [WebBaseLoader(url).load() for url in urls]

In [ ]:
# Print the first 40 characters of the third document's content
print(docs[2][0].page_content.strip()[:40])

In [ ]:
# Split the fetched documents into smaller chunks for indexing into our vectorstore

# Flatten the list of documents
docs_list = [item for sublist in docs for item in sublist]

# recursively split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000, chunk_overlap=200
)

# Split the documents into smaller chunks
doc_splits = text_splitter.split_documents(docs_list)

# Retriever Tool
- Index documents into a vector store.
- Tools are utilities designed to be called by a model: their inputs are designed to be generated by models, and their outputs are designed to be passed back to models.

In [ ]:
# Create the vectorstore using in-memory vector store and OpenAI embeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding = embeddings
)
retriever = vectorstore.as_retriever()

In [ ]:
# use langchain inbuilt retriever tool for tool creation
retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_blog_posts",
    "Search and return information about Lilian Weng blog posts.",
)

In [ ]:
retriever_tool.invoke({"query": "types of reward hacking"})

# Generate query
- Start building the components (nodes and edges) for our agentic RAG graph.
- Components will operate on the MessagesState — graph state that contains a messages key with a list of chat messages.

- **`generate_query_or_respond` node** — This node calls an LLM to generate a response based on the current graph state (list of messages). Given the input messages, the model can decide to either retrieve information using the previously created `retriever_tool` or respond directly to the user. The chat model is given access to the `retriever_tool` via `.bind_tools`, allowing it to choose the best action for each query

In [ ]:
def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    response = (
        llm
        .bind_tools([retriever_tool]).invoke(state["messages"])
    )
    return {"messages": [response]}

In [ ]:
input = {"messages": [{"role": "user", "content": "hello!"}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

In [ ]:
#semantic search query
input = {
    "messages": [
        {
            "role": "user",
            "content": "What does Lilian Weng say about types of reward hacking?",
        }
    ]
}
generate_query_or_respond(input)["messages"][-1].pretty_print()

# 📑 Grade Documents

- **`grade_documents`** — This function evaluates whether the retrieved documents are relevant to the user's question. It uses a model with a structured output schema (`GradeDocuments`) to perform document grading. Based on the grading decision, the function returns the next node to proceed to: either `generate_answer` (if relevant) or `rewrite_question

`with_structured_output` is a crucial feature in LangChain that enforces structured, predictable responses from language models through:

Key Benefits
- Schema Validation: Enforces responses to match a predefined Pydantic model
- Type Safety: Ensures all fields have correct data types
- Error Prevention: Eliminates common issues like:
    - Missing required fields
    - Invalid enum values
    - Hallucinated fields
    - Malformed JSON responses

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

#define the grading prompt
GRADE_PROMPT = (
    "You are a grader assessing relevance of a retrieved document to a user question. \n "
    "Here is the retrieved document: \n\n {context} \n\n"
    "Here is the user question: {question} \n"
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n"
    "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."
)


class GradeDocuments(BaseModel):
    
    """Model to grade documents based on relevance to a question."""
    
    binary_score: str = Field(
        description="A binary score indicating whether the document is relevant to the question. Relevance score: 'yes' if relevant, or 'no' if not relevant"
    )

def grade_documents(state: MessagesState) -> Literal["generate_answer", "rewrite_question"]:
    """Determine whether the retrieved documents are relevant to the question."""
    
    question = state["messages"][0].content
    context = state["messages"][1].content
    
    #get the prompt
    prompt = GRADE_PROMPT.format(context=context, question=question)

    #invoke the model
    response = (llm.with_structured_output(GradeDocuments).invoke(
        [{"role": "user", "content": prompt}]
    ))
    
    #get the score
    score = response.binary_score
    if score == "yes":
        return "generate_answer"
    else:
        return "rewrite_question"

In [ ]:
from langchain_core.messages import convert_to_messages

#test the function 
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {"role": "tool", "content": "meow", "tool_call_id": "1"},
        ]
    )
}
grade_documents(input)

In [ ]:
input = {
"messages": convert_to_messages(
    [
        {
            "role": "user",
            "content": "What does Lilian Weng say about types of reward hacking?",
        },
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "1",
                    "name": "retrieve_blog_posts",
                    "args": {"query": "types of reward hacking"},
                }
            ],
        },
        {
            "role": "tool",
            "content": "reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering",
            "tool_call_id": "1",
        },
    ]
)
}
grade_documents(input)

# Rewrite question
- This node helps improve search results by reformulating the user's original question when retrieved documents are deemed irrelevant.

In [ ]:
REWRITE_PROMPT = (
    "Look at the input and try to reason about the underlying semantic intent / meaning.\n"
    "Here is the initial question:"
    "\n ------- \n"
    "{question}"
    "\n ------- \n"
    "Formulate an improved question:"
)


def rewrite_question(state: MessagesState):
    """Rewrite the original user question."""
    messages = state["messages"]
    question = messages[0].content
    prompt = REWRITE_PROMPT.format(question=question)
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"messages": [{"role": "user", "content": response.content}]}

In [ ]:
# test the functionality

input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {"role": "tool", "content": "meow", "tool_call_id": "1"},
        ]
    )
}

response = rewrite_question(input)
print(response["messages"][-1]["content"])

# Generate an Answer
- `generate_answer` node: if we pass the grader checks, we can generate the final answer based on the original question and the retrieved context

In [ ]:
GENERATE_PROMPT = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \n"
    "Context: {context}"
)


def generate_answer(state: MessagesState):
    """Generate an answer."""
    question = state["messages"][0].content
    context = state["messages"][-1].content
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = llm.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}

In [ ]:
# test the generate answer functionality
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "retrieve_blog_posts",
                        "args": {"query": "types of reward hacking"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "reward hacking can be categorized into two types: environment or goal misspecification, and reward tampering",
                "tool_call_id": "1",
            },
        ]
    )
}

response = generate_answer(input)
response["messages"][-1].pretty_print()

# Assemble the Graph

- Start with a `generate_query_or_respond` and determine if we need to call retriever_tool
- Route to next step using tools_condition:
    - If `generate_query_or_respond` returned `tool_calls`, call `retriever_tool` to retrieve context
    - Otherwise, respond directly to the user
- Grade retrieved document content for relevance to the question (grade_documents) and route to next step:
    - If not relevant, rewrite the question using `rewrite_question` and then call `generate_query_or_respond` again
    - If relevant, proceed to `generate_answer` and generate final response using the `ToolMessage` with the retrieved document context

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

#define the graph workflow
workflow = StateGraph(MessagesState)

# Define the nodes we will cycle between
workflow.add_node(generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retriever_tool]))
workflow.add_node(rewrite_question)
workflow.add_node(generate_answer)

# Define the edges (flow) between nodes
workflow.add_edge(START, "generate_query_or_respond")

# Decide whether to retrieve
workflow.add_conditional_edges("generate_query_or_respond", 
    # Assess LLM decision (call `retriever_tool` tool or respond to the user)                               
                               tools_condition, 
                               {
                                   "tools" : "retrieve",
                                   END: END
                               })
workflow.add_conditional_edges("retrieve", grade_documents) # Assess agent decision
workflow.add_edge("generate_answer", END)
workflow.add_edge("rewrite_question", "generate_query_or_respond")

# Compile
graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import convert_to_messages

for chunk in graph.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What does Lilian Weng say about types of reward hacking?",
            }
        ]
    }
):
    for node, update in chunk.items():
        print("\n=== Update from node:", node, "===")
        # Get the last message from the update
        message = update["messages"][-1]
        
        # Handle both dict and AIMessage formats
        if isinstance(message, dict):
            print("Role:", message["role"])
            print("Content:", message["content"])
            if "tool_calls" in message:
                print("Tool Calls:", message["tool_calls"])
        else:
            print("Role:", message.type)
            print("Content:", message.content)
            if hasattr(message, "additional_kwargs") and "tool_calls" in message.additional_kwargs:
                print("Tool Calls:", message.additional_kwargs["tool_calls"])
        
        print("-" * 50)